# 시제 전환 연구를 위한 개수 생성 함수.

- 시제 전환 연구를 위해서 각 문장의 맨 끝 용언구의 유사보조용언 연쇄 마지막 어절에 '-었-'이 존재하는지 확인해서 문장 시제열을 만듦
- 이전 문장/다음 문장이 존재하는지, 이전 문장과 다음 문장의 문장 시제도 함께 저장.
- 문장 추출 함수 있음.

### log
- 2026.05.05 
    - 입력: "..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.7.csv"
    - 출력:  "..\csv\통계용\세종_문어_문장끝_인용제외_body만_통계_20260505_10-48.csv"
    - 필터 적용: FILTER_BODY_SEN_ENDS_NOT_QUOTE = lambda df: df[(df['sent_end_V'] == True) & (df['sent_end_V_in_quote'] == False) & (df['speaker'] == 'body')]



In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
START = datetime.now()

#대상 파일 읽어오기.
CSV_PATH = Path(r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.7.csv")
df_word = pd.read_csv(CSV_PATH)
print(f"*** 파일 읽기 완료: {CSV_PATH} ({len(df_word):,}행): {START.strftime("%Y.%m.%d_%H:%M:%S")}, during: {(datetime.now()-START)} ***")
df_word.columns

*** 파일 읽기 완료: ..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver9.7.csv (12,973,652행): 2026.05.20_16:25:55, during: 0:00:36.547663 ***


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id'],
      dtype='object')

In [2]:
#각종 함수 선언 셀, process_dataframe, generate_pivot_statistics, expand_window
import numpy as np
import gc
from pandas.api.types import is_categorical_dtype
import re
from pathlib import Path
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0]   # 상위폴더를 ROOT로 넣음.
sys.path.append(str(ROOT))
from stats.v_affix_attach import v_affix_attach, add_v_no_by_merge_and_fallback
from stats.en_no_fix import fix_en_number_with_merge, make_en_no_sub #EN_No 수정

# 결과에 덧붙이는 정보 처리 함수
def process_dataframe(result_df: pd.DataFrame) -> pd.DataFrame:
    # ---- 기본 방어: Series가 들어오면 DF로 ----
    if isinstance(result_df, pd.Series):
        result_df = result_df.reset_index(name=result_df.name or "ID")

    # ---- 인덱스가 의미있는 경우만 풀기 (MultiIndex/Named index) ----
    if not isinstance(result_df.index, pd.RangeIndex) or result_df.index.name is not None:
        result_df = result_df.reset_index()

    def add_ep_flags(df: pd.DataFrame, form_col: str, prefix: str) -> pd.DataFrame:
        if form_col in df.columns and f'{prefix}_T' not in df.columns:
            re_tt = re.compile(r'(?:었었|았었)')
            re_t  = re.compile(r'(?:었|았)')
            re_m  = re.compile(r'(?:겠|겄)')

            s = df[form_col].astype('string').str.replace(' + ', '', regex=False)
            df[form_col] = s
            tt = s.str.contains(re_tt, na=False)
            df[f'{prefix}_TT'] = tt
            df[f'{prefix}_T']  = (~tt) & s.str.contains(re_t, na=False)
            df[f'{prefix}_M']  = s.str.contains(re_m, na=False)
        return df

    # EN No sub정보 덧붙이기
    if ("EN_form" in result_df.columns) and ("EN_No_sub" not in result_df.columns):
        result_df = make_en_no_sub(result_df)

    # EP / f_EP / sentence / pre / post
    result_df = add_ep_flags(result_df, 'EP_form', 'EP')
    result_df = add_ep_flags(result_df, 'f_EP_form', 'f_EP')
    result_df = add_ep_flags(result_df, 'sentence_f_EP_form', 'sentence_f_EP')
    result_df = add_ep_flags(result_df, 'prev_sentence_f_EP_form', 'prev_sentence_f_EP')
    result_df = add_ep_flags(result_df, 'next_sentence_f_EP_form', 'next_sentence_f_EP')
            
    # V_No 생성
    if 'V_form_0' in result_df.columns and 'V_No' not in result_df.columns:
        V_No_file = r"..\..\label\000.V_No_(2026).csv"
        result_df = add_v_no_by_merge_and_fallback(result_df, V_No_file, encoding='utf-8-sig')

    return result_df

#개선된 그룹바이 함수
def pivot_statistics(
    df: pd.DataFrame,
    index_columns,
    filter_fn=None,
):
    """
    단일 DF에서 index_columns 조합별 빈도(size)를 계산해 Series로 반환.
    - dropna=False: 키 NA 포함
    - observed=True: category 조합 폭발 방지
    - sort=False: 속도
    """
    if filter_fn is not None:
        df = filter_fn(df)

    try:
        s = (
            df.groupby(index_columns, dropna=False, observed=True, sort=False)
              .size()
              .rename("ID")
        )
    except TypeError:
        # dropna 파라미터 호환 이슈 대비 (구버전/특정 상황)
        SENTINEL = "__<NA>__"
        sub = df[index_columns].copy()
        for c in index_columns:
            sub[c] = sub[c].where(sub[c].notna(), SENTINEL)
        s = sub.value_counts(sort=False).rename("ID")
        s.index = pd.MultiIndex.from_tuples(
            [tuple(np.nan if x == SENTINEL else x for x in tup) for tup in s.index],
            names=index_columns
        )
    return s

# Null값 등으로 인해서 제외되는 값이 있는지 확인하는 함수
def debug_drop_reason(df, index_columns):
    n_all = len(df)
    n_key_na = df[index_columns].isna().any(axis=1).sum()     # 키에 NA 있는 행 수
    n_id_na = df['ID'].isna().sum() if 'ID' in df.columns else 0

    # pivot_table식 count가 실제로 세는 개수(대략적인 감산식)
    approx_pivot_count = df.loc[~df[index_columns].isna().any(axis=1), 'ID'].notna().sum()

    # 권장 경로: groupby + size (키 NA 포함)
    gb_size = df.groupby(index_columns, dropna=False, observed=True).size().sum()

    return {
        "len(df)": int(n_all),
        "키에 NA 있는 행": int(n_key_na),
        "ID가 NA인 행": int(n_id_na),
        "pivot_table(count) 합": int(approx_pivot_count),
        "groupby(size, dropna=False) 합": int(gb_size),
    }

#개수 출력함수: main
def pivot_statistics_to_df(
    df: pd.DataFrame,
    index_columns,
    filter_fn=None,
    process_fn=None,
) -> pd.DataFrame:
    s = pivot_statistics(df, index_columns, filter_fn=filter_fn)
    out = s.reset_index()  # columns: index_columns + ['ID']

    # ✅ 후처리는 여기서 DF에 직접 적용 (결과 작아진 뒤)
    if process_fn is not None:
        out = process_fn(out)

    return out

from datetime import datetime
print(f"함수 생성 완료 - {datetime.now().strftime("%Y.%m.%d_%H:%M:%S")}")

함수 생성 완료 - 2026.05.20_16:31:42


In [3]:
df = df_word.copy()
df.columns

Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id'],
      dtype='object')

In [4]:
# ---------------------------------------------
# f_EN_no, f_EN_no_sub, f_EN_label 생성, V0_form, V0_label, V0_No 생성
# ---------------------------------------------
START = datetime.now()
print(f"시작 - {START.strftime("%Y%m%d_%H:%M:%S")}")

# ---------------------------------------------
# 1. f_en_no, f_en_no_sub, f_en_label 생성
# ---------------------------------------------
if True: 
    df = df.drop(columns=["f_EN_form", "f_EN_label", "f_EN_No"], errors="ignore") # 기존에 있으면 제거 (중복 방지)

    # -1이면 자기 word_id2로 대체
    df["f_word_id"] = df["f_word_id"].where(
        df["f_word_id"] != -1,
        df["word_id"]
    )

    lookup = (
        df[
            [
                "sen_id",
                "word_id",
                "EN_form",
                "EN_No",
                "EN_No_sub",
                "EN_label",
                "EP_form"
            ]
        ]
        .rename(
            columns={
                "word_id": "f_word_id",
                "EN_form": "f_EN_form",
                "EN_No": "f_EN_No",
                "EN_No_sub": "f_EN_No_sub",
                "EN_label": "f_EN_label",
                "EP_form": "f_EP_form"
            }
        )
    )

    # ------------------------------------------
    # sen_id + f_word_id 기준 self merge
    # ------------------------------------------
    df = df.merge(
        lookup,
        on=["sen_id", "f_word_id"],
        how="left"
    )

print(f"f_EN 생성 완료 - {datetime.now()-START}")
# ---------------------------------------------
# 2. V0_form, V0_label, V0_No 생성
# ---------------------------------------------
START = datetime.now()
print(f"시작 - {START.strftime("%Y%m%d_%H:%M:%S")}")

if True:
    df = df.drop(columns=["V0_form", "V0_label", "V0_No"], errors="ignore") # 기존에 있으면 제거 (중복 방지)

    df["V0_word_id"] = df["V0_word_id"].where(
        df["V0_word_id"] != -1,
        df["word_id"]
    )

    lookup_V = (
        df[
            [
                "sen_id",
                "word_id",
                "V_form",
                "V_label",
                "V_No"
            ]
        ]
        .rename(
            columns={
                "word_id": "V0_word_id",
                "V_form": "V0_form",
                "V_label": "V0_label",
                "V_No": "V0_No"
            }
        )
    )

    df = df.merge(
        lookup_V,
        on=["sen_id", "V0_word_id"],
        how="left"
    )

print(f"V0 생성 완료 - {datetime.now()-START}")
df.columns

시작 - 20260520_16:31:48
f_EN 생성 완료 - 0:00:16.769254
시작 - 20260520_16:32:05
V0 생성 완료 - 0:00:24.872546


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id', 'f_EN_form', 'f_EN_No', 'f_EN_No_sub',
       'f_EN_label', 'f_EP_form', 'V0_form', 'V0_label', 'V0_No'],
      dtype='object')

In [5]:
# 문장별 시제 정보 붙이기
# ==============================================
# # sentence 기준으로 prev/next 문장 시제 정보 붙이기: prev/next 문장 번호, 문장 ID, 문장끝 동사구 정보 붙이기
# 옵션1. 
# USE_NOT_QUOTE_CHAIN: - 인용문 문장은 아예 제외한 df_not_quote를 따로 만들어  그 집합 안에서만 prev / next 계산
# 옵션2.
# BODY_ONLY_FOR_SENTENCE_LINKS = True   # True면 body 문장만 기준
# ==============================================

import pandas as pd
from datetime import datetime

START = datetime.now()
print(f"시작 - {START.strftime('%Y%m%d_%H:%M:%S')}")

# -------------------------------------------------------------
# 옵션
# -------------------------------------------------------------
BODY_ONLY_FOR_SENTENCE_LINKS = True   # True면 body 문장만 기준
USE_NOT_QUOTE_CHAIN = True            # True면 인용문 문장을 제외한 체인으로 prev/next 계산

# -------------------------------------------------------------
# 문장 파일 읽기
# -------------------------------------------------------------
SEN_CSV = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen.csv"
df_sen = pd.read_csv(SEN_CSV, low_memory=False)
print(f"***sentence 파일 읽기 완료: {SEN_CSV} ({len(df_sen):,}행): {datetime.now().strftime('%Y%m%d_%H-%M-%S')} ***")

# -----------------------------------
# 0. df_sen에서 sen_num / rev_sen_num / speaker 가져오기
# -----------------------------------
sen_cols = ["sen_id", "sen_num", "rev_sen_num", "speaker"]

existing_sen_cols = [c for c in ["sen_num", "rev_sen_num", "speaker"] if c in df.columns]
if existing_sen_cols:
    df = df.drop(columns=existing_sen_cols)

df = df.merge(
    df_sen[sen_cols].drop_duplicates(subset=["sen_id"]),
    on="sen_id",
    how="left"
)

# -----------------------------------
# 1. 문장끝 동사구 대표값 추출
#    sent_end_V=True인 것 중 첫 번째 값 사용
# -----------------------------------
sen_end_info_all = (
    df.loc[df["sent_end_V"] == True,
           ["docu_id", "sen_id", "sen_num", "rev_sen_num", "speaker",
            "f_EP_form", "sent_end_V_in_quote"]]
      .dropna(subset=["docu_id", "sen_id", "sen_num"])
      .sort_values(["docu_id", "sen_num"])
      .drop_duplicates(subset=["docu_id", "sen_id"], keep="first")
      .rename(columns={
          "f_EP_form": "sentence_f_EP_form",
          "sent_end_V_in_quote": "sentence_sent_end_V_in_quote"
      })
      .copy()
)
# -----------------------------------
# 2. target sentence filter
# -----------------------------------
df_target_sentence = sen_end_info_all.copy()

# 2.1 인용문 제외용 df_target_sentence 만들기
#    sentence_sent_end_V_in_quote == False 인 문장만 남김
if USE_NOT_QUOTE_CHAIN:
    df_target_sentence = df_target_sentence.loc[
        df_target_sentence["sentence_sent_end_V_in_quote"] == False
    ].copy()

# 2.2 body 문장만 남기는 df_target_sentence 만들기
#    speaker == "body" 인 문장만 남김
if BODY_ONLY_FOR_SENTENCE_LINKS:
    df_target_sentence = df_target_sentence.loc[
        df_target_sentence["speaker"] == "body"
    ].copy()


# -----------------------------------
# 3. prev/next 계산용 문장 기준표 만들기
#    -> df_target_sentence 기준
# -----------------------------------
sentence_index = (
    df_target_sentence[["docu_id", "sen_id", "sen_num", "rev_sen_num"]]
    .dropna(subset=["docu_id", "sen_id", "sen_num"])
    .drop_duplicates(subset=["docu_id", "sen_id"])
    .sort_values(["docu_id", "sen_num"])
    .copy()
)

# -----------------------------------
# 4. 이전 / 다음 문장 번호, 문장 ID 계산
#    -> 인용문 제외된 체인 안에서만 계산됨
# -----------------------------------
sentence_index["prev_sen_num"] = sentence_index.groupby("docu_id")["sen_num"].shift(1)
sentence_index["next_sen_num"] = sentence_index.groupby("docu_id")["sen_num"].shift(-1)
sentence_index["prev_sen_id"] = sentence_index.groupby("docu_id")["sen_id"].shift(1)
sentence_index["next_sen_id"] = sentence_index.groupby("docu_id")["sen_id"].shift(-1)

sentence_index["has_prev_sentence"] = sentence_index["prev_sen_id"].notna()
sentence_index["has_next_sentence"] = sentence_index["next_sen_id"].notna()


# -----------------------------------
# 5. 현재 문장 정보 붙이기
# -----------------------------------
sentence_index = sentence_index.merge(
    df_target_sentence[["docu_id", "sen_id", "sentence_f_EP_form", "sentence_sent_end_V_in_quote"]],
    on=["docu_id", "sen_id"],
    how="left"
)

# -----------------------------------
# 6. 이전 문장 정보 붙이기
# -----------------------------------
prev_info = df_target_sentence[
    ["docu_id", "sen_id", "sentence_f_EP_form", "sentence_sent_end_V_in_quote"]
].rename(columns={
    "sen_id": "prev_sen_id",
    "sentence_f_EP_form": "prev_sentence_f_EP_form",
    "sentence_sent_end_V_in_quote": "prev_sentence_sent_end_V_in_quote"
})

sentence_index = sentence_index.merge(
    prev_info,
    on=["docu_id", "prev_sen_id"],
    how="left"
)

# -----------------------------------
# 7. 다음 문장 정보 붙이기
# -----------------------------------
next_info = df_target_sentence[
    ["docu_id", "sen_id", "sentence_f_EP_form", "sentence_sent_end_V_in_quote"]
].rename(columns={
    "sen_id": "next_sen_id",
    "sentence_f_EP_form": "next_sentence_f_EP_form",
    "sentence_sent_end_V_in_quote": "next_sentence_sent_end_V_in_quote"
})

sentence_index = sentence_index.merge(
    next_info,
    on=["docu_id", "next_sen_id"],
    how="left"
)

# -----------------------------------
# 8. 기존 컬럼 제거
# -----------------------------------
cols_to_drop = [
    "prev_sen_num",
    "next_sen_num",
    "prev_sen_id",
    "next_sen_id",
    "has_prev_sentence",
    "has_next_sentence",
    "sentence_f_EP_form",
    "prev_sentence_f_EP_form",
    "next_sentence_f_EP_form",
    "sentence_sent_end_V_in_quote",
    "prev_sentence_sent_end_V_in_quote",
    "next_sentence_sent_end_V_in_quote",
]

existing_cols_to_drop = [c for c in cols_to_drop if c in df.columns]
if existing_cols_to_drop:
    df = df.drop(columns=existing_cols_to_drop)

# -----------------------------------
# 9. 원래 df에 붙이기
#     주의: quote 문장은 sentence_index에 없으므로
#     그 문장들의 prev_/next_는 NaN으로 남는다.
# -----------------------------------
df = df.merge(
    sentence_index[
        [
            "docu_id",
            "sen_id",
            "sen_num",
            "rev_sen_num",
            "prev_sen_num",
            "next_sen_num",
            "prev_sen_id",
            "next_sen_id",
            "has_prev_sentence",
            "has_next_sentence",
            "sentence_f_EP_form",
            "prev_sentence_f_EP_form",
            "next_sentence_f_EP_form",
            "sentence_sent_end_V_in_quote",
            "prev_sentence_sent_end_V_in_quote",
            "next_sentence_sent_end_V_in_quote",
        ]
    ],
    on=["docu_id", "sen_id", "sen_num", "rev_sen_num"],
    how="left"
)

print(
    f"merge 완료 - {START.strftime('%Y%m%d_%H-%M-%S')}, "
    f"during: {datetime.now() - START}, "
    f"BODY_ONLY_FOR_SENTENCE_LINKS={BODY_ONLY_FOR_SENTENCE_LINKS}, "
    f"USE_NOT_QUOTE_CHAIN={USE_NOT_QUOTE_CHAIN}"
)

df.columns

시작 - 20260520_16:32:40
***sentence 파일 읽기 완료: ..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.8_sen.csv (1,048,127행): 20260520_16-32-45 ***
merge 완료 - 20260520_16-32-40, during: 0:00:31.444999, BODY_ONLY_FOR_SENTENCE_LINKS=True, USE_NOT_QUOTE_CHAIN=True


Index(['ID', 'file_id', 'docu_id', 'sen_id', 'word_id', 'rev_word_id',
       'form/label', 'N_form', 'N_label', 'V_No', 'V_form', 'V_label',
       'V_form_old', 'V_label_old', 'EP_form', 'EP_label', 'EN_form',
       'EN_label', 'J_form', 'J_label', 'EN_No', 'EN_No_sub', 'sent_end',
       'sent_end_V', 'sent_end_V_in_quote', 'quote_type', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_word_id', 'f_word_id', 'f_EN_form', 'f_EN_No', 'f_EN_No_sub',
       'f_EN_label', 'f_EP_form', 'V0_form', 'V0_label', 'V0_No', 'sen_num',
       'rev_sen_num', 'speaker', 'prev_sen_num', 'next_sen_num', 'prev_sen_id',
       'next_sen_id', 'has_prev_sentence', 'has_next_sentence',
       'sentence_f_EP_form', 'prev_sentence_f_EP_form',
       'next_sentence_f_EP_form', 'sentence_sent_end_V_in_quote',
       'prev_sentence_sent_end_V_in_quote',
       'next_sentence_sent_end_V_in_quote'],
      dtype='object')

In [ ]:
#document 파일 읽기, 필요한 파일 병합
DOCU_CSV = r"..\csv\original_csv\세종문어_document_정보_ver.1.1.csv"
df_docu = pd.read_csv(DOCU_CSV, low_memory=False)

#file 파일 읽기
FILE_CSV = r"..\csv\original_csv\세종문어_file_정보_ver.1.1.csv"
df_file = pd.read_csv(FILE_CSV, low_memory=False)

# ---
# document 정보에서 필요한 컬럼 병합
# ---
def safe_merge(df, df_add, key, cols):
    cols_to_use = [key] + [c for c in cols if c not in df.columns]
    return df.merge(df_add[cols_to_use], on=key, how="left")

df = safe_merge(df, df_sen, "sen_id", ["speaker"])
df = safe_merge(
    df,
    df_docu,
    "docu_id",
    ['category', '매체', '내용', '파일제목', '저자', '출판사', '출판연도',
     'head', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio']
)

print(f"merge 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")

df['문서범주'] = df['category'].map({
    '보도해설':'신문', 
    '사설': '신문', 
    '칼럼': '신문', 
    '인문사회': '책', 
    '체험기술': '체험', 
    '허구일반': '허구', 
    '자연': '책', 
    '총류': '책', 
    '허구아동': '허구'
})

df.columns
print(f"파일 읽기 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}")
print(f"df_file: {df_file.columns.tolist()}")
print(f"df_docu: {df_docu.columns.tolist()}")
print(f"df_file: {len(df_file):,}행, df_docu: {len(df_docu):,}행, df_sen: {len(df_sen):,}행, df: {len(df):,}행")

파일 읽기 완료 - 2026.05.20_16:34:16
df_file: ['file_id', '매체', '내용', '내용2', '파일제목', '저자', '출판사', '출판연도', '구어/문어', '분류기호', '분류기호2', '내용3', '분류기호4', 'sent_count', 'head_count', 'body_count', 'body_has_E_count', 'body_not_quote_count', 'body_not_quote_and_었_count', '었_결합_오즈비', '었_결합_로그오즈비', '었_결합_등급', '었_결합_성향']
df_docu: ['docu_id', 'file_id', 'docu_num', 'category', '매체', '내용', '내용2', '파일제목', '저자', '출판사', '출판연도', 'head', '제목', '구어/문어', '분류기호', '분류기호2', '내용3', '분류기호4', 'sen_count', 'sen_count_has_E', 'sen_count_not_quote', 'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No', 'dominant_EN_label', 'dominant_count', 'dominant_ratio', 'sent_count', 'head_count', 'body_count', 'body_has_E_count', 'body_not_quote_count', 'body_not_quote_and_었_count', '었_결합_오즈비', '었_결합_로그오즈비', '었_결합_등급', '었_결합_성향']
df_file: 369행, df_docu: 33,155행, df_sen: 1,048,127행, df: 12,975,418행


In [8]:
print(f"총 행: {len(df):,}, N 수: {df['N_label'].count():,},  V 수: {df['V_label'].count():,}  EN 수: {df['EN_label'].count():,}")
print(f"V0수 : {df[(df['V_label'].notna()) & (df['VX0_order']==-1)]['ID'].count():,},       f_en수 : {df[(df['EN_label'].notna()) & (df['VX_len']==0)]['ID'].count():,}")

print("\n문장 끝")
print(f"문장의 마지막 V0수 : {df[(df['V_label'].notna()) & (df['VX0_order']==-1) & (df['sent_end_V']==True)]['ID'].count():,}, 문장의 마지막 f_en수 : {df[(df['EN_label'].notna()) & (df['VX_len']==0)& (df['sent_end_V']==True)]['ID'].count():,}")

print('\nbody만')
print(f"문장의 마지막 V0수 : {df[(df['V_label'].notna()) & (df['VX0_order']==-1) & (df['sent_end_V']==True)& (df['speaker']=='body')]['ID'].count():,}, 문장의 마지막 f_en수 : {df[(df['EN_label'].notna()) & (df['VX_len']==0)& (df['sent_end_V']==True)& (df['speaker']=='body')]['ID'].count():,}")

print('\nbody, has_prev_sentence')
print(f"문장의 마지막 V0수 : {df[(df['V_label'].notna()) & (df['VX0_order']==-1) & (df['sent_end_V']==True)& (df['speaker']=='body') &(df['has_prev_sentence'] == True)]['ID'].count():,}, 문장의 마지막 f_en수 : {df[(df['EN_label'].notna()) & (df['VX_len']==0)& (df['sent_end_V']==True)& (df['speaker']=='body')&(df['has_prev_sentence'] == True)]['ID'].count():,}")

print('\nbody, has_next_sentence')
print(f"문장의 마지막 V0수 : {df[(df['V_label'].notna()) & (df['VX0_order']==-1) & (df['sent_end_V']==True)& (df['speaker']=='body') &(df['has_next_sentence'] == True)]['ID'].count():,}, 문장의 마지막 f_en수 : {df[(df['EN_label'].notna()) & (df['VX_len']==0)& (df['sent_end_V']==True)& (df['speaker']=='body')&(df['has_next_sentence'] == True)]['ID'].count():,}")

총 행: 12,975,418, N 수: 5,051,557,  V 수: 4,585,517  EN 수: 4,599,152
V0수 : 3,817,063,       f_en수 : 3,832,248

문장 끝
문장의 마지막 V0수 : 979,425, 문장의 마지막 f_en수 : 982,446

body만
문장의 마지막 V0수 : 963,071, 문장의 마지막 f_en수 : 966,066

body, has_prev_sentence
문장의 마지막 V0수 : 795,466, 문장의 마지막 f_en수 : 797,739

body, has_next_sentence
문장의 마지막 V0수 : 795,445, 문장의 마지막 f_en수 : 797,725


In [9]:
OUT_DIR = r"..\csv\통계용"
PREFIX = "세종_문어_문장끝_인용제외_body만_통계"
stamp = datetime.now().strftime("%Y%m%d_%H-%M")#파일 저장용 stamp

index_columns = ['문서범주', 'category','매체', 'file_id', 'docu_id', 'speaker',
                 'sen_count', 'sen_count_has_E', 'sen_count_not_quote',
       'sen_count_has_E_not_quote', 'base_count_not_quote', 'dominant_EN_No',
       'dominant_EN_label', 'dominant_count', 'dominant_ratio',
       'V_No', 'V_form',
       'V_label', 'EP_form', 'EN_form', 'EN_label', 'EN_No', 'EN_No_sub', 'VX_len',
       'Next_VX_No', 'Next_VX_form', 'VX0_No', 'VX0_form', 'VX0_order',
       'V0_form', 'V0_No', 'V0_label',
       'f_EP_form', 'f_EN_form', 'f_EN_No',
       'f_EN_No_sub', 'f_EN_label',
       'has_prev_sentence', 'has_next_sentence',
       'sentence_f_EP_form', 'prev_sentence_f_EP_form',
       'next_sentence_f_EP_form', 
       'sentence_sent_end_V_in_quote', 'prev_sentence_sent_end_V_in_quote',
       'next_sentence_sent_end_V_in_quote']

FILTER_SEN_ENDS = lambda df: df[(df['sent_end_V'] == True)]
FILTER_SEN_ENDS_NOT_QUOTE = lambda df: df[(df['sent_end_V'] == True) & (df['sent_end_V_in_quote'] == False)]
FILTER_BODY_SEN_ENDS_NOT_QUOTE = lambda df: df[(df['sent_end_V'] == True) & (df['sent_end_V_in_quote'] == False) & (df['speaker'] == 'body')]
#에러 확인
if True:
    print(debug_drop_reason(df, index_columns))

#실행
result_df = pivot_statistics_to_df(
    df =df,
    index_columns=index_columns,
    process_fn=process_dataframe,
    filter_fn=FILTER_BODY_SEN_ENDS_NOT_QUOTE
)

# 결과 컬럼 순서 재배치 및 컬럼명 변경
if True:
    front_cols = []
    last_cols = ['ID' ]
    rest_cols = [col for col in result_df.columns if col not in front_cols + last_cols]
    result_df = result_df[front_cols + rest_cols + last_cols]

    # 합계 컬럼명 변경
    result_df = result_df.rename(columns={
        "ID": "count",
    })

print(f"result_df length: {len(result_df):,}, now: {datetime.now().strftime("%Y.%m.%d_%H:%M:%S")}")
result_df.columns

# 결과 DataFrame 저장
save_folder = Path(OUT_DIR) 
output_file_name = save_folder / f"{PREFIX}_{stamp}.csv"
output_file_name.parent.mkdir(parents=True, exist_ok=True)  # 폴더 생성

result_df.to_csv(output_file_name, index=True, encoding='utf-8-sig')
print(f"*** 저장 완료: {output_file_name} ({len(result_df):,}행) ***")

{'len(df)': 12975418, '키에 NA 있는 행': 12975027, 'ID가 NA인 행': 0, 'pivot_table(count) 합': 391, 'groupby(size, dropna=False) 합': 12975418}
result_df length: 1,022,399, now: 2026.05.20_16:37:07
*** 저장 완료: ..\csv\통계용\세종_문어_문장끝_인용제외_body만_통계_20260520_16-36.csv (1,022,399행) ***


In [20]:
#문장 저장
FILTER_SEN_ENDS = lambda df: df[(df['sent_end_V'] == True)]
FRONT_COLS = ['문서범주', 'category', '파일제목', 'sentence_form']

from datetime import datetime
from pathlib import Path
import re

def attach_sentence_form(df, df_sen,
        filter = FILTER_SEN_ENDS, 
        front_cols = []):
    # 1. 필터 적용
    df_filtered = filter(df)

    # 2. 필요한 컬럼만 준비
    sen_lookup = df_sen[['sen_id', 'sentence_form']].drop_duplicates()

    # 3. 병합
    result = df_filtered.merge(
        sen_lookup,
        on='sen_id',
        how='left'
    )

    # 4. 원하는 컬럼 순서 앞으로 배치
    remaining_cols = [c for c in result.columns if c not in front_cols]

    result = result[front_cols + remaining_cols]

    return result

def save_csv_by_column(df, group_col, output_dir=OUT_DIR, prefix=None):
    """
    특정 컬럼 값별로 df를 나누어 CSV로 저장한다.
    예: group_col='문서범주'
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    now = datetime.now().strftime("%Y%m%d_%H%M%S")
    prefix = prefix or group_col

    for value, df_part in df.groupby(group_col, dropna=False):
        # 파일명에 쓰기 어려운 문자 제거
        safe_value = str(value) if value == value else "NA"
        safe_value = re.sub(r'[\\/:*?"<>|]', "_", safe_value)

        filename = f"{prefix}_{safe_value}_{now}.csv"
        filepath = output_dir / filename

        df_part.to_csv(filepath, index=False, encoding="utf-8-sig")

        print(f"저장 완료: {filepath} ({len(df_part):,}행) ")
    

result_with_sentence = attach_sentence_form(df, df_sen, FILTER_SEN_ENDS, 
                                         FRONT_COLS)
print(f"문장 정보 붙이기 완료 - {datetime.now().strftime('%Y.%m.%d_%H:%M:%S')}, during: {(datetime.now()-START)}")
result_with_sentence.head(5)

save_csv_by_column(result_with_sentence, "문서범주", output_dir=OUT_DIR, prefix=f"{PREFIX}_문장포함")

문장 정보 붙이기 완료 - 2026.05.05_11:12:14, during: 0:47:06.561490
저장 완료: ..\csv\통계용\세종_문어_문장끝_인용제외_body만_통계_문장포함_신문_20260505_111214.csv (300,799행) 
저장 완료: ..\csv\통계용\세종_문어_문장끝_인용제외_body만_통계_문장포함_책_20260505_111214.csv (468,468행) 
저장 완료: ..\csv\통계용\세종_문어_문장끝_인용제외_body만_통계_문장포함_체험_20260505_111214.csv (111,869행) 
저장 완료: ..\csv\통계용\세종_문어_문장끝_인용제외_body만_통계_문장포함_허구_20260505_111214.csv (456,187행) 
